# 🏛️ CreditRisk-IFRS9: Enterprise Credit Risk Modeling & Accounting Impairment
**Author:** Gabriel Proaño  
**Standard Governance:** IFRS 9 Financial Instruments, BCBS Basel III/IV, ECOA / FCRA  

---

## 1. Executive Summary & Mathematical Framework
This notebook provides the foundational methodology for:
1. **Regulatory Scorecard Engineering:** Weight of Evidence ($WoE$) & Information Value ($IV$)
2. **Reject Inference:** Correcting sample selection bias on unobservable rejected applicants
3. **Probability of Default (PD):** WoE Logistic Regression vs. Calibrated LightGBM
4. **Multi-Horizon Survival PD:** Term structure $S(t)$ and marginal default intensities
5. **IFRS 9 Three-Stage Impairment:** Forward-looking Vasicek Point-in-Time (PIT) expected credit loss ($ECL$)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.generator import CreditPortfolioGenerator
from src.features.scorecard import ScorecardTransformer
from src.models.pd_engine import ProbabilityOfDefaultEngine
from src.models.reject_inference import RejectInferenceEngine
from src.models.survival_pd import SurvivalPDEngine
from src.simulation.ecl_engine import IFRS9ECLEngine

print('✓ All core modules imported successfully.')

## 2. Portfolio Simulation & Underwriting Selection
We simulate a representative retail credit portfolio with standard risk covariates:

In [ ]:
gen = CreditPortfolioGenerator(random_seed=42)
portfolio = gen.generate(n_samples=3000, reject_rate=0.25)
acc_df = portfolio.accepted_loans
rej_df = portfolio.rejected_applications

print('Total applications:', len(portfolio.all_applications))
print('Accepted loans:', len(acc_df), '| 12m Default Rate:', round(acc_df['default_12m'].mean() * 100, 2), '%')
print('Rejected applications:', len(rej_df))
acc_df.head()

## 3. Scorecard Engineering: WoE & Information Value (IV)
$$\text{WoE}_i = \ln\left( \frac{\text{DistGood}_i}{\text{DistBad}_i} \right), \quad \text{IV} = \sum_i (\text{DistGood}_i - \text{DistBad}_i) \times \text{WoE}_i$$

In [ ]:
feature_cols = ['bureau_score', 'debt_to_income', 'delinquencies_2yrs', 'revolving_utilization', 'annual_income']
sc = ScorecardTransformer(target_score=600, target_odds=50, pdo=20)
sc.fit(acc_df[feature_cols], acc_df['default_12m'])
X_woe = sc.transform_woe(acc_df[feature_cols])

iv_df = pd.DataFrame([vars(s) for s in sc.iv_summary])
iv_df

## 4. Probability of Default & Probability Calibration
Comparing Regulatory WoE Logistic Regression vs. Calibrated LightGBM:

In [ ]:
pd_eng = ProbabilityOfDefaultEngine(random_state=42)
pd_eng.fit(X_woe, acc_df[feature_cols], acc_df['default_12m'], calibration_method='isotonic')
m_log, m_lgb = pd_eng.evaluate(X_woe, acc_df[feature_cols], acc_df['default_12m'])

intercept, coefs = pd_eng.get_logistic_coefficients()
sc.set_model_weights(intercept, coefs)
scorecard_table = sc.get_scorecard_table()
scorecard_table.head(10)

## 5. IFRS 9 Forward-Looking Expected Credit Loss (ECL)
Discounted three-stage provisions under Vasicek systemic macroeconomic scenarios:

In [ ]:
calibrated_pds = pd_eng.predict_pd_logistic(X_woe, calibrated=True)
ecl_eng = IFRS9ECLEngine(effective_interest_rate=0.065)
ecl_df, ecl_summary = ecl_eng.process_portfolio(acc_df, calibrated_pds)

print('Total Exposure (EAD): $', f'{ecl_summary.total_exposure:,.2f}')
print('Total Weighted ECL : $', f'{ecl_summary.total_weighted_ecl:,.2f}')
print('Portfolio Coverage :', f'{ecl_summary.total_coverage_ratio*100:.2f}%')
print('Stage 1:', ecl_summary.stage1_count, '| Stage 2 (SICR):', ecl_summary.stage2_count, '| Stage 3 (Default):', ecl_summary.stage3_count)